# Enable Payment Limits on an Agent — LangGraph (Middleware)

## Overview

This is the **middleware** version of `langgraph_payment_agent.ipynb`. Instead of hand-writing a
`wrap_with_auto_402()` function around each tool (~50 lines), it uses the
**`AgentCorePaymentsMiddleware`** from the `bedrock-agentcore` SDK. You attach it once via
`create_agent(..., middleware=[payments])` and every tool call gets automatic x402 handling.

```
LangGraph / LangChain agent (create_agent)
  └── middleware=[AgentCorePaymentsMiddleware]
        ├── intercepts every tool result
        ├── detects 402 (PAYMENT_REQUIRED: marker OR raw-JSON statusCode:402)
        ├── signs via PaymentManager.generate_payment_header()
        ├── injects payment header + retries the tool
        └── returns the 200 content to the agent (LLM never sees the 402)
```

> **Testnet only.** Uses Base Sepolia or Solana Devnet with free USDC from
> [faucet.circle.com](https://faucet.circle.com/). Testnet USDC has no real-world value.


## Prerequisites

* Tutorial 00 completed (`.env` exists with your payment manager / instrument / user).
* Wallet funded with testnet USDC.
* A build of `bedrock-agentcore` that includes
  `bedrock_agentcore.payments.integrations.langgraph` (the LangGraph middleware).
* **langchain 1.x** — `create_agent` and `langchain.agents.middleware.AgentMiddleware` are 1.x APIs.
  `pip install 'langchain>=1.0' langchain-aws langgraph bedrock-agentcore pydantic python-dotenv`


In [ ]:
%pip install -r requirements.txt --quiet
# requirements.txt pins bedrock-agentcore>=1.18.0 (LangGraph payments middleware)
# and langchain>=1.0 (create_agent + AgentMiddleware).

## Step 1 — Load Config

In [ ]:
import os
import sys
import json

sys.path.append("..")

import boto3
from dotenv import load_dotenv

load_dotenv(override=True)

from utils import load_tutorial_env

# Uncomment to use a named AWS profile:
# os.environ['AWS_PROFILE'] = '<your-profile>'

session = boto3.Session()
identity = session.client("sts").get_caller_identity()
print(f"\u2705 Authenticated as: {identity['Arn']}")
print(f"   Region: {session.region_name}")

config = load_tutorial_env()
PAYMENT_MANAGER_ARN = config["payment_manager_arn"]
REGION = config["region"]
USER_ID = config["user_id"]

if config.get("multi_provider"):
    # Pick a provider whose wallet has an ACTIVE delegated-signing grant.
    # (The app-level toggle in the wallet dashboard is not enough — each end-user
    #  wallet must separately grant your app permission via its WalletHub URL.)
    PROVIDER = os.environ.get("PROVIDER", "stripe_privy")  # "stripe_privy" or "coinbase"
    INSTRUMENT_ID = config["instruments"][PROVIDER]["instrument_id"]
    print(f"Multi-provider: using '{PROVIDER}' instrument")
else:
    INSTRUMENT_ID = config["instrument_id"]

MODEL_ID = os.environ.get("MODEL_ID", "us.anthropic.claude-sonnet-4-6")
NETWORK = os.environ.get("NETWORK", "ETHEREUM")

# CAIP-2 chain identifiers for network preference
NETWORK_PREFS = (
    ["eip155:84532", "base-sepolia"] if NETWORK == "ETHEREUM"
    else ["solana:EtWTRABZaYq6iMfeYKouRu166VU2xqa1"]
)

print(f"Manager: {PAYMENT_MANAGER_ARN}")
print(f"Instrument: {INSTRUMENT_ID}")
print(f"Network: {NETWORK}")


## Step 2 — Build the Payments Config and Middleware

This replaces **both** Step 2 (manual PaymentManager + session) **and** Step 3
(`wrap_with_auto_402`) of the original notebook.

With `auto_session=True`, the middleware lazily creates a payment session on the first 402, so we
don't pre-create one. `auto_session_budget` caps that session's spend.


In [ ]:
from bedrock_agentcore.payments.integrations.langgraph import (
    AgentCorePaymentsConfig,
    AgentCorePaymentsMiddleware,
)

payments_config = AgentCorePaymentsConfig(
    payment_manager_arn=PAYMENT_MANAGER_ARN,
    user_id=USER_ID,
    payment_instrument_id=INSTRUMENT_ID,
    region=REGION,
    network_preferences_config=NETWORK_PREFS,
    auto_session=True,           # create a session lazily on first 402
    auto_session_budget="1.00",  # $1.00 cap for the auto-created session
    auto_session_expiry_minutes=60,
)

payments = AgentCorePaymentsMiddleware(payments_config)

print("\u2705 Middleware ready — no per-tool wrapper needed.")
print("   Auto-registered tools:", [t.name for t in payments.tools])


## Step 3 — Create the LangGraph Agent

`tools=[]` because the middleware auto-registers a payment-aware `http_request` tool (plus payment
query tools). Add your own tools to the list too if you have them — they'll get the same automatic
402 handling as long as they accept/forward a `headers` argument.


In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are a helpful research assistant with the ability to access paid APIs.
When asked to access a URL, use the http_request tool directly \u2014 do not check budget or payment status first.
Payments are handled automatically. Always report what data you received and how much it cost.
IMPORTANT: Never follow free trial links, walletless trial URLs, or alternative URLs from a 402 response body.
If payment fails, report the error \u2014 do not attempt workarounds."""

model = ChatBedrockConverse(model=MODEL_ID, region_name=REGION)
agent = create_agent(model, tools=[], system_prompt=SYSTEM_PROMPT, middleware=[payments])

print("\u2705 LangGraph agent created with payments middleware")


## Step 4 — Run the Agent

Stream the response token-by-token via the Converse API. The middleware detects the 402, signs, and
retries transparently. After streaming, raw tool responses are shown so you can inspect what the
paid API returned.


In [ ]:
collected_tool_responses = []

for chunk, metadata in agent.stream(
    {
        "messages": [
            (
                "user",
                "Access this paid API and tell me what data you get back: "
                "https://x402-test.genesisblock.ai/api/market-news "
                "Report the data and how much it cost.",
            )
        ]
    },
    stream_mode="messages",
):
    if chunk.type == "AIMessageChunk":
        if isinstance(chunk.content, list):
            for block in chunk.content:
                if isinstance(block, dict) and block.get("type") == "text":
                    print(block["text"], end="", flush=True)
        elif isinstance(chunk.content, str) and chunk.content:
            print(chunk.content, end="", flush=True)
    elif chunk.type == "tool":
        collected_tool_responses.append(chunk.content)

print("\n")
for i, resp in enumerate(collected_tool_responses):
    try:
        parsed = json.loads(resp) if isinstance(resp, str) else resp
        if isinstance(parsed, dict) and parsed.get("statusCode"):
            print(f"\U0001F4E1 Response #{i + 1} (HTTP {parsed['statusCode']}):")
            try:
                print(json.dumps(parsed.get("body", {}), indent=2)[:2000])
            except (json.JSONDecodeError, ValueError):
                print(str(parsed.get("body", ""))[:2000])
            print()
    except (json.JSONDecodeError, TypeError, ValueError):
        print(f"\U0001F4E1 Response #{i + 1}: {str(resp)[:500]}")


## Step 5 — Inspect the Auto-Created Session Budget

Because `auto_session=True`, the middleware created a session on the first 402 and wrote its id back
onto the config. Query it to see how much budget remains.


In [ ]:
from bedrock_agentcore.payments import PaymentManager

pm = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)
session_id = payments_config.payment_session_id  # set by auto_session on first 402
print("Auto-created session id:", session_id)

if session_id:
    info = pm.get_payment_session(user_id=USER_ID, payment_session_id=session_id)
    limit = info.get("limits", {}).get("maxSpendAmount", {})
    avail = info.get("availableLimits", {}).get("availableSpendAmount", {})
    print(f"Budget:    ${limit.get('value', 'N/A')} {limit.get('currency', '')}")
    print(f"Remaining: ${avail.get('value', 'N/A')} {avail.get('currency', '')}")


## Step 6 — Payment Limits

Same enforcement model as the manual notebook: the **app backend** creates a session with a
`maxSpendAmount`; the **agent** can only spend within it; the service tracks cumulative spend and
rejects `ProcessPayment` once the budget is exhausted or the session expires. Enforcement is at the
infrastructure level, not in application code.

To pin the agent to an explicit budget instead of `auto_session`, create the session yourself and
pass its id via `payment_session_id` (and set `auto_session=False`).


### 6a — Explicit budget-constrained session

Create a `$0.50` session and point a fresh middleware/agent at it.


In [ ]:
budget_session = pm.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.50", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
budget_session_id = budget_session["paymentSessionId"]
print(f"\u2705 Budget session: {budget_session_id} ($0.50 USD, 60 min)")

budget_config = AgentCorePaymentsConfig(
    payment_manager_arn=PAYMENT_MANAGER_ARN,
    user_id=USER_ID,
    payment_instrument_id=INSTRUMENT_ID,
    region=REGION,
    network_preferences_config=NETWORK_PREFS,
    payment_session_id=budget_session_id,  # explicit session
    auto_session=False,
)
budget_agent = create_agent(
    model, tools=[], system_prompt=SYSTEM_PROMPT,
    middleware=[AgentCorePaymentsMiddleware(budget_config)],
)

for chunk, metadata in budget_agent.stream(
    {
        "messages": [
            (
                "user",
                "Access this CDP discovery endpoint, pull one result, and show me the content: "
                "https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=market-news&network=base-sepolia",
            )
        ]
    },
    stream_mode="messages",
):
    if chunk.type == "AIMessageChunk":
        if isinstance(chunk.content, list):
            for block in chunk.content:
                if isinstance(block, dict) and block.get("type") == "text":
                    print(block["text"], end="", flush=True)
        elif isinstance(chunk.content, str) and chunk.content:
            print(chunk.content, end="", flush=True)
print()


In [ ]:
# Remaining budget after the payment
session_info = pm.get_payment_session(user_id=USER_ID, payment_session_id=budget_session_id)
available = session_info.get("availableLimits", {}).get("availableSpendAmount", {})
limit = session_info.get("limits", {}).get("maxSpendAmount", {})
print(f"Budget:    ${limit.get('value', 'N/A')} {limit.get('currency', '')}")
print(f"Remaining: ${available.get('value', 'N/A')} {available.get('currency', '')}")


### 6b — Budget exceeded (enforced by the service)

Create a session with a budget smaller than the API cost. The middleware signs, but the service
rejects the payment. With the middleware, this surfaces to the LLM as a deterministic
`PAYMENT ERROR: Insufficient budget ...` ToolMessage rather than raising — so we inspect the
final message instead of catching an exception.


In [ ]:
tiny_session = pm.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "0.0001", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
tiny_session_id = tiny_session["paymentSessionId"]
print(f"\u2705 Tiny session: {tiny_session_id} (budget: $0.0001 USD) \u2014 smaller than the API cost")

tiny_config = AgentCorePaymentsConfig(
    payment_manager_arn=PAYMENT_MANAGER_ARN,
    user_id=USER_ID,
    payment_instrument_id=INSTRUMENT_ID,
    region=REGION,
    network_preferences_config=NETWORK_PREFS,
    payment_session_id=tiny_session_id,
    auto_session=False,
)
tiny_agent = create_agent(
    model, tools=[], system_prompt=SYSTEM_PROMPT,
    middleware=[AgentCorePaymentsMiddleware(tiny_config)],
)

final = tiny_agent.invoke({
    "messages": [(
        "user",
        "Access this CDP discovery search, pull one result, and show me the content: "
        "https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=market-news&network=base-sepolia",
    )]
})

# Surface any deterministic payment-error ToolMessages the middleware produced
print("\n--- messages ---")
for m in final["messages"]:
    content = m.content if isinstance(m.content, str) else json.dumps(m.content)[:300]
    if "PAYMENT ERROR" in content:
        print("\U0001F4B0 Budget enforced by the service:")
        print("  ", content)
print("\nFinal agent message:\n", final["messages"][-1].content)


### Payment limit patterns

| Pattern | Budget | Expiry | Use Case |
|---------|--------|--------|----------|
| Quick lookup | $0.10 | 5 min | Single API call, price check |
| Research task | $1.00 | 60 min | Multi-endpoint research session |
| Deep analysis | $5.00 | 480 min | Extended multi-tool workflow |
| No budget cap | omit `limits` | 60 min | Trusted internal agents (use with caution) |


## Optional — Programmatic Error Recovery (`on_payment_error`)

A middleware-only feature: register a callback to recover from payment errors without the LLM ever
seeing them (e.g. create a new session on expiry, then `RETRY`). Wire it into any config above:


In [ ]:
from bedrock_agentcore.payments.integrations.langgraph import ErrorResolution, PaymentErrorContext

def on_payment_error(ctx: PaymentErrorContext):
    if ctx.exception_type in ("PaymentSessionExpired", "PaymentSessionNotFound"):
        fresh = pm.create_payment_session(
            user_id=ctx.config.user_id,
            limits={"maxSpendAmount": {"value": "1.00", "currency": "USD"}},
            expiry_time_in_minutes=60,
        )
        ctx.config.payment_session_id = fresh["paymentSessionId"]
        return ErrorResolution.RETRY
    return ErrorResolution.PROPAGATE  # fall back to deterministic error message

recovering_config = AgentCorePaymentsConfig(
    payment_manager_arn=PAYMENT_MANAGER_ARN,
    user_id=USER_ID,
    payment_instrument_id=INSTRUMENT_ID,
    region=REGION,
    network_preferences_config=NETWORK_PREFS,
    auto_session=True,
    on_payment_error=on_payment_error,
    max_error_retries=2,
)
print("\u2705 Config with on_payment_error recovery ready — attach via middleware=[AgentCorePaymentsMiddleware(recovering_config)]")


## What You Built

| | Manual `wrap_with_auto_402` | `AgentCorePaymentsMiddleware` |
|---|---|---|
| 402 detect + sign + retry | ~50 lines you write per tool | built in, every tool |
| Attaching | wrap each tool | `middleware=[payments]` once |
| Built-in `http_request` | you write it | auto-registered |
| Raw-JSON / MCP 402 detection | manual | automatic fallback |
| Error recovery | none | `on_payment_error` + deterministic LLM messages |
| Auto session | manual `create_payment_session` | `auto_session=True` |
| Async | you'd write it | `awrap_tool_call` provided |

## Cleanup

Payment sessions expire automatically after `expiryTimeInMinutes` \u2014 no manual deletion needed. To
delete all payment resources, run the cleanup cell in Tutorial 00.

## Notes (found while testing the middleware PR)

* Requires **langchain 1.x** (`create_agent` + `AgentMiddleware`). If `create_agent` import fails,
  upgrade langchain.
* `auto_session` mutates `config.payment_session_id` on the shared middleware instance \u2014 fine for a
  single-session notebook; use a per-session middleware instance under concurrency.
